# 11 — V2 model experiments: three candidate families, strong baselines, feature ablation, chronological evaluation

**สถานะ: งานวิจัยเท่านั้น | production_ready = false | operational = BLOCKED.** ไม่มีการรับประกันว่าโมเดลจะชนะ baseline

- อินพุต: ผล notebook 10 ที่ Codex ตรวจรับแล้ว (`reports/v2/preparation/preparation_manifest.json` + `data/v2/*.csv.gz`, อ่านอย่างเดียว ตรวจ hash) และ
  `workflow/model-protocol-v2-lock.json` ที่ manager สร้าง (schema `roadaware-v2-experiment-lock/1`) — ถ้าไม่มีหรือ hash ไม่ตรง notebook จะหยุดก่อน fit
- config: `config/model_experiment_v2.json` (ค่าตาม `workflow/model-protocol-v2-draft.md` ไม่เปลี่ยน, ไม่มี tuning loop)
- หน่วย daily และ 6h แยกกันทั้งหมด; 4 fold ขยายตามเวลา; fit เฉพาะเซลล์ Train ที่ target_end_local <= เวลา issue (วันก่อนหน้า 18:00 +07)
- เลือกจาก pooled OOF เท่านั้น แล้วเขียน `selection_lock.json` **ก่อน** อ่าน diagnostic 2024/2025 และตรวจซ้ำหลังจากนั้น
- เอาต์พุต: `reports/v2/models/*` (aggregate, tracked) และ `data/v2/models/*` (prediction รายแถว + bundle, gitignored)
- การวิเคราะห์ error ละเอียด/กราฟ อยู่ใน notebook 12 (งานถัดไป)

Run All ใน VS Code ด้วย kernel `.venv` หรือ `.venv/bin/python backend/scripts/run_v2_notebooks.py --stage model`

In [ ]:
# SETUP: bound numerical thread pools BEFORE numpy / scikit-learn are imported in this kernel (RF n_jobs=2 is in the config)
import os
for _k in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[_k] = '2'
print({k: os.environ[k] for k in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS')})

In [ ]:
# FUNCTIONS (notebook 11, part 1: I/O + hashes, experiment-lock / dependency verification, feature guard, unit frames, issue-time folds) - self-contained; tests execute both FUNCTIONS cells in order (no network, no fit in this cell)
import datetime as dt, gzip, hashlib, json, math, os, sys, time, warnings
from pathlib import Path
import numpy as np, pandas as pd

KEY_SEP = '\x1f'
PANEL_REQUIRED = ['province_key', 'target_date', 'y_recorded_count', 'training_eligible', 'split', 'issue_time_local', 'target_start_local',
                  'target_end_local']
TS_FORMAT = '%Y-%m-%dT%H:%M:%S%z'


class DependencyError(RuntimeError):
    """An input is missing, malformed or differs from the hash its producer recorded: stop, never guess."""


class LockError(RuntimeError):
    """The experiment lock or the selection lock is absent, malformed or does not match the actual bytes: stop before any fit/diagnostic."""


# ---------- generic I/O ----------
def utc_now():
    return dt.datetime.now(dt.timezone.utc).replace(microsecond=0).isoformat()


def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()


def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()


def json_safe(o):
    """Strict-JSON friendly copy: NaN/inf/NaT/NA -> None, numpy scalars -> Python, dates -> ISO strings."""
    if o is None or o is pd.NaT or o is pd.NA:
        return None
    if isinstance(o, dict):
        return {str(k): json_safe(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, set)):
        return [json_safe(v) for v in o]
    if isinstance(o, (bool, np.bool_)):
        return bool(o)
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, (float, np.floating)):
        return float(o) if math.isfinite(float(o)) else None
    if isinstance(o, (pd.Timestamp, dt.datetime, dt.date)):
        return o.isoformat()
    if isinstance(o, Path):
        return str(o)
    return o


def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(json_safe(obj), indent=2, ensure_ascii=False, allow_nan=False) + '\n', 'utf-8')
    os.replace(tmp, path)
    return path


def read_json_strict(path):
    return json.loads(Path(path).read_text('utf-8'), parse_constant=lambda c: (_ for _ in ()).throw(ValueError(f'{path}: non-strict JSON {c}')))


def write_csv(path, df):
    """Plain CSV, LF line endings; any infinite numeric value is refused (finite aggregates only)."""
    path = Path(path)
    num = df.select_dtypes(include=[np.number])
    if len(num.columns) and np.isinf(num.to_numpy(dtype=float)).any():
        raise ValueError(f'{path.name}: infinite values')
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, lineterminator='\n')
    return path


def write_csv_gz(path, df):
    """Gzip CSV with a fixed header (mtime 0, no stored name): identical tables give identical bytes; floats written with repr precision."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    with open(tmp, 'wb') as fh, gzip.GzipFile(filename='', mode='wb', fileobj=fh, mtime=0) as gz:
        gz.write(df.to_csv(index=False, lineterminator='\n', float_format='%.17g').encode('utf-8'))
    os.replace(tmp, path)
    return path


def verify_file(path, expected_sha256, label):
    """sha256 of an input must equal the hash recorded by its producer; raises DependencyError otherwise."""
    p = Path(path)
    if not p.is_file():
        raise DependencyError(f'{label}: missing input {p}')
    got = sha256_file(p)
    if not expected_sha256 or got != expected_sha256:
        raise DependencyError(f'{label}: sha256 {got[:12]} != recorded {str(expected_sha256)[:12]} ({p}); input changed since its producer ran')
    return got


def functions_cells_source(nb_path):
    """Concatenated source of every '# FUNCTIONS' code cell in notebook order (hashed into manifests / the selection lock)."""
    nb = json.loads(Path(nb_path).read_text('utf-8'))
    srcs = [''.join(c['source']) if isinstance(c['source'], list) else c['source'] for c in nb['cells'] if c['cell_type'] == 'code']
    hits = [s for s in srcs if s.lstrip().startswith('# FUNCTIONS')]
    if not hits:
        raise ValueError(f'{nb_path}: no FUNCTIONS cell')
    return '\n\n'.join(hits)


def code_cells_sha256(nb_path):
    """sha256 of every code cell source (cell index:id -> hash)."""
    nb = json.loads(Path(nb_path).read_text('utf-8'))
    out = {}
    for i, c in enumerate(nb['cells']):
        if c['cell_type'] == 'code':
            s = ''.join(c['source']) if isinstance(c['source'], list) else c['source']
            out[f"{i}:{c.get('id', '')}"] = sha256_bytes(s.encode('utf-8'))
    return out


# ---------- roots ----------
def find_backend():
    """ROADAWARE_BACKEND_ROOT, else the first cwd ancestor that is (or contains) backend/ with config/model_experiment_v2.json + notebooks/."""
    env = os.environ.get('ROADAWARE_BACKEND_ROOT')
    if env:
        return Path(env).expanduser().resolve()
    here = Path.cwd().resolve()
    for d in (here, *here.parents):
        for cand in (d, d / 'backend'):
            if (cand / 'config' / 'model_experiment_v2.json').is_file() and (cand / 'notebooks').is_dir():
                return cand
    raise FileNotFoundError('backend root not found from the working directory; set ROADAWARE_BACKEND_ROOT')


def env_path(name, default):
    v = os.environ.get(name)
    return Path(v).expanduser().resolve() if v else Path(default).resolve()


def inside(p, root):
    p, root = Path(p).resolve(), Path(root).resolve()
    return p == root or root in p.parents


def check_write_roots(report_dir, model_data, prep_dir, prep_data, read_only, prep_data_files):
    """Write roots may not overlap the read-only 10 report dir / other read-only roots; the model data root may sit INSIDE the 10 data
    root (data/v2/models) but may not equal/contain it and may not contain any accepted 10 data output."""
    for label, w in (('model report dir', report_dir), ('model data root', model_data)):
        for rl, r in [('10 preparation dir', prep_dir)] + list(read_only):
            if inside(w, r) or inside(r, w):
                raise ValueError(f'{label} {w} overlaps read-only {rl} {r}')
    if inside(prep_data, model_data):
        raise ValueError(f'model data root {model_data} equals or contains the read-only 10 data root {prep_data}')
    if inside(report_dir, prep_data) or inside(prep_data, report_dir):
        raise ValueError(f'model report dir {report_dir} overlaps the 10 data root {prep_data}')
    hit = [str(f) for f in prep_data_files if inside(f, model_data)]
    if hit:
        raise ValueError(f'model data root contains accepted 10 outputs {hit[:3]}')
    return True


COMPLETED_MARKERS = ('selection_lock.json', 'model_manifest.json', 'oof_metrics_pooled.csv', 'fit_log.csv')
MODEL_DATA_MARKERS = ('oof_predictions_*.csv.gz', 'diagnostic_predictions_*.csv.gz', 'bundles')


def existing_results(report_dir, model_data):
    """Artifacts of an earlier notebook 11 run in these roots (old locks/results are preserved, never overwritten)."""
    hits = [Path(report_dir) / n for n in COMPLETED_MARKERS if (Path(report_dir) / n).exists()]
    for pat in MODEL_DATA_MARKERS:
        hits += sorted(Path(model_data).glob(pat))
    return [str(h) for h in hits]


# ---------- experiment lock / preparation manifest ----------
def verify_preparation_manifest(m, icfg):
    if m.get('schema') != icfg['preparation_manifest_schema'] or m.get('producer') != icfg['preparation_producer']:
        raise DependencyError(f"preparation manifest schema/producer {m.get('schema')}/{m.get('producer')} unexpected")
    if m.get('status') not in icfg['accepted_preparation_status']:
        raise DependencyError(f"preparation status {m.get('status')!r} not accepted")
    if m.get('production_ready') is not False or m.get('operational_status') != 'BLOCKED':
        raise DependencyError('preparation manifest must stay research-only (production_ready false, BLOCKED)')
    it = m.get('next_stage_interface') or {}
    for unit in ('daily', '6h'):
        need = {'panel', 'features', 'keys', 'target', 'mask', 'split', 'feature_columns'}
        if not need <= set(it.get(unit, {})):
            raise DependencyError(f'next_stage_interface.{unit} lacks {sorted(need - set(it.get(unit, {})))}')
    if not {'feature_groups', 'categorical', 'never_model_inputs', 'splits'} <= set(it):
        raise DependencyError('next_stage_interface lacks feature_groups/categorical/never_model_inputs/splits')
    return True


def verify_experiment_lock(lock_path, protocol_path, config_path, prep_manifest_path, prep_manifest, schema, fields):
    """Manager-created planning lock (written only after Codex accepted 09/10): every recorded hash must equal the actual bytes."""
    p = Path(lock_path)
    if not p.is_file():
        raise LockError(f'experiment lock {p} not found: Codex creates it only after accepting 09/10 and locking input hashes; no fit without it')
    lock = read_json_strict(p)
    missing = [k for k in fields if k not in lock]
    if missing:
        raise LockError(f'experiment lock lacks {missing}')
    if lock['schema'] != schema:
        raise LockError(f"experiment lock schema {lock['schema']!r} != {schema!r}")
    for key, path in (('protocol_sha256', protocol_path), ('model_config_sha256', config_path), ('preparation_manifest_sha256', prep_manifest_path)):
        got = sha256_file(path)
        if lock[key] != got:
            raise LockError(f'{key}: lock {str(lock[key])[:12]} != actual {got[:12]} ({path})')
    want = sorted(((o['path'], o['sha256']) for o in prep_manifest['data_outputs']))
    have = sorted(((o.get('path'), o.get('sha256')) for o in lock['preparation_data_outputs']))
    if have != want:
        raise LockError('lock preparation_data_outputs differ from the accepted 10 manifest data_outputs')
    run10 = prep_manifest.get('run_id', prep_manifest.get('started_utc'))
    if str(lock['preparation_run_id']) != str(run10):
        raise LockError(f"preparation_run_id {lock['preparation_run_id']!r} != 10 manifest run {run10!r}")
    if not isinstance(lock['source_run_id'], str) or not lock['source_run_id']:
        raise LockError('source_run_id must be a non-empty string')
    return lock


def check_source_run_id(lock, prep_manifest, sources_dir):
    """Cross-check source_run_id against the 09 manifest only while it still has the hash 10 recorded (09 may legitimately rerun later)."""
    rec = [i for i in prep_manifest.get('inputs', []) if Path(str(i.get('path', ''))).name == 'sources_manifest.json']
    p = Path(sources_dir) / 'sources_manifest.json'
    if not rec or not p.is_file():
        return {'status': 'not_checked_09_manifest_unavailable'}
    if sha256_file(p) != rec[0]['sha256']:
        return {'status': 'not_checked_09_manifest_changed_after_10'}
    run09 = read_json_strict(p).get('run_id', read_json_strict(p).get('started_utc'))
    if str(run09) != lock['source_run_id']:
        raise LockError(f"source_run_id {lock['source_run_id']!r} != 09 manifest run {run09!r} recorded by the accepted 10 run")
    return {'status': 'verified', 'source_run_id': run09}


# ---------- feature guard ----------
def validate_feature_columns(cols, approved, forbidden_names, forbidden_substrings, never=()):
    """Every model input must be in the accepted 10 allowlist and pass the forbidden name/substring guard (IDs, dates, targets, masks)."""
    bad = []
    if len(set(cols)) != len(cols):
        bad.append(('duplicate columns', sorted({c for c in cols if list(cols).count(c) > 1})))
    for c in cols:
        if c not in approved:
            bad.append((c, 'not in approved allowlist'))
        if c in forbidden_names or c in never:
            bad.append((c, 'forbidden name'))
        hit = [s for s in forbidden_substrings if s in c]
        if hit:
            bad.append((c, f'forbidden substring {hit}'))
    if bad:
        raise ValueError(f'feature guard rejected: {bad}')
    return True


def variant_columns(unit_cols, feature_groups, variants, extra_groups):
    """Columns per fixed variant, in the 10 allowlist order; every allowed column belongs to exactly one group and 'full' uses all of them."""
    group_of = {}
    for g, cs in feature_groups.items():
        for c in cs:
            if c in group_of:
                raise ValueError(f'{c} in two feature groups')
            group_of[c] = g
    orphan = [c for c in unit_cols if c not in group_of]
    if orphan:
        raise ValueError(f'unit features without a group: {orphan}')
    out = {v: [c for c in unit_cols if group_of[c] in set(gs) | set(extra_groups)] for v, gs in variants.items()}
    for v, cs in out.items():
        if not cs:
            raise ValueError(f'variant {v} has no columns')
    return out


# ---------- unit frames ----------
def parse_bool_strict(s, label):
    m = s.astype(str).map({'True': True, 'False': False, 'true': True, 'false': False})
    if m.isna().any():
        raise DependencyError(f'{label}: non-boolean values {sorted(set(s[m.isna()].astype(str)))[:5]}')
    return m.astype(bool)


def parse_local_ts(s, offset, label):
    if not s.astype(str).str.endswith(offset).all():
        raise DependencyError(f'{label}: every timestamp must carry the explicit {offset} offset')
    return pd.to_datetime(s, format=TS_FORMAT)


def load_unit_frame(panel_path, features_path, it, cat_cols, offset):
    """Panel + accepted 10 features joined on identical keys (row-by-row equal, no duplicates). Returns a frame with keys, y, eligible,
    split, tz-aware issue/start/end, bl_weekday/bl_month (baseline-only, from target_date) and every allowed feature (NaN kept)."""
    keys, feats = list(it['keys']), list(it['feature_columns'])
    pan = pd.read_csv(panel_path, dtype=str, keep_default_na=False)
    need = list(dict.fromkeys(PANEL_REQUIRED + keys + [it['target'], it['mask'], it['split']]))
    miss = [c for c in need if c not in pan.columns]
    if miss:
        raise DependencyError(f'{Path(panel_path).name}: missing columns {miss}')
    str_cols = list(dict.fromkeys(keys + [c for c in cat_cols if c in feats]))
    fe = pd.read_csv(features_path, dtype={c: str for c in str_cols}, keep_default_na=False, na_values=[''])
    fmiss = [c for c in keys + feats if c not in fe.columns]
    if fmiss:
        raise DependencyError(f'{Path(features_path).name}: missing columns {fmiss}')
    if len(fe) != len(pan) or not (fe[keys].to_numpy() == pan[keys].to_numpy()).all():
        raise DependencyError('feature keys differ from panel keys (row order must match exactly)')
    if pan.duplicated(keys).any():
        raise DependencyError('duplicate panel keys')
    y = pd.to_numeric(pan[it['target']], errors='raise')
    if (y < 0).any() or not np.array_equal(y.to_numpy(float), np.floor(y.to_numpy(float))):
        raise DependencyError('target must be a non-negative integer count')
    out = pan[keys].copy()
    out['y'] = y.astype(float).to_numpy()
    out['eligible'] = parse_bool_strict(pan[it['mask']], it['mask']).to_numpy()
    out['split'] = pan[it['split']].to_numpy()
    out['start_ts'] = parse_local_ts(pan['target_start_local'], offset, 'target_start_local')
    out['end_ts'] = parse_local_ts(pan['target_end_local'], offset, 'target_end_local')
    out['issue_ts'] = parse_local_ts(pan['issue_time_local'], offset, 'issue_time_local')
    td = pd.to_datetime(out['target_date'], format='%Y-%m-%d')
    out['bl_weekday'] = td.dt.dayofweek.astype(str).to_numpy()
    out['bl_month'] = td.dt.month.astype(str).to_numpy()
    for c in feats:
        if c in keys:
            continue  # e.g. time_block: the feature IS the key column (asserted equal above)
        if c in cat_cols:
            out[c] = fe[c].to_numpy()
        else:
            v = pd.to_numeric(fe[c], errors='raise').astype(float).to_numpy()
            if np.isinf(v).any():
                raise DependencyError(f'{c}: infinite feature values')
            out[c] = v
    return out


# ---------- issue-time folds ----------
def issue_time_for(date_str, issue_hour, days_before, offset):
    d = dt.date.fromisoformat(date_str) - dt.timedelta(days=days_before)
    return pd.Timestamp(f'{d.isoformat()}T{issue_hour:02d}:00:00{offset}')


def fit_mask(frame, cutoff, train_split):
    """Cells allowed for fitting: Train split, eligible (known) label, and the whole target interval ended by the cutoff issue time."""
    return frame['eligible'].to_numpy() & frame['split'].eq(train_split).to_numpy() & (frame['end_ts'] <= cutoff).to_numpy()


def eval_mask(frame, start, end, split=None):
    m = frame['eligible'].to_numpy() & (frame['target_date'] >= start).to_numpy() & (frame['target_date'] <= end).to_numpy()
    if split is not None:
        m &= frame['split'].eq(split).to_numpy()
    return m


def key_tuples(frame, mask, keys):
    return set(map(tuple, frame.loc[mask, keys].to_numpy()))


def check_issue_times(frame, issue_hour, days_before, offset):
    """Panel integrity: every cell's issue time equals target_date - days_before at issue_hour local (identical for all blocks of a day)."""
    want = pd.to_datetime(frame['target_date'] + f'T{issue_hour:02d}:00:00{offset}', format=TS_FORMAT) - pd.Timedelta(days=days_before)
    if not (frame['issue_ts'] == want).all():
        raise DependencyError('panel issue_time_local disagrees with target_date - 1 day 18:00 local')
    return True


def check_fold(frame, train_m, eval_m, issue, keys, name):
    """Hard boundary checks BEFORE any fit: non-empty, train ends <= issue, every evaluation issue >= the fold issue, train targets end
    before evaluation targets start, disjoint keys."""
    if not train_m.any() or not eval_m.any():
        raise ValueError(f'{name}: empty train or evaluation cohort')
    tr, ev = frame.loc[train_m], frame.loc[eval_m]
    if not (tr['end_ts'] <= issue).all():
        raise ValueError(f'{name}: a training cell ends after the issue time')
    if not (ev['issue_ts'] >= issue).all():
        raise ValueError(f'{name}: an evaluation issue time precedes the fold issue {issue}')
    if tr['end_ts'].max() > ev['start_ts'].min():
        raise ValueError(f'{name}: training targets overlap the evaluation period')
    if (train_m & eval_m).any() or key_tuples(frame, train_m, keys) & key_tuples(frame, eval_m, keys):
        raise ValueError(f'{name}: train and evaluation keys are not disjoint')
    return {'fold': name, 'issue_time_local': issue.isoformat(), 'train_first_target_date': tr['target_date'].min(),
            'train_last_target_date': tr['target_date'].max(), 'train_max_target_end_local': tr['end_ts'].max().isoformat(),
            'eval_first_target_date': ev['target_date'].min(), 'eval_last_target_date': ev['target_date'].max(),
            'n_train': int(train_m.sum()), 'n_eval': int(eval_m.sum()), 'train_y_sum': float(tr['y'].sum()), 'eval_y_sum': float(ev['y'].sum()),
            'disjoint_keys': True, 'train_end_le_issue': True}

In [ ]:
# FUNCTIONS (notebook 11, part 2: Train-only baselines, train-only preprocessing, the three candidate families, OOF loop, metrics, selection, stability gate, calendar-block bootstrap, selection lock, bundles/reload) - self-contained
import datetime as dt, json, math, time, warnings
from pathlib import Path
import numpy as np, pandas as pd

KEY_SEP = '\x1f'


# ---------- Train-only baselines (JSON-serialisable states; no labels outside Train are ever read) ----------
def _keystr(frame, cols):
    if not cols:
        return pd.Series([''] * len(frame), index=frame.index)
    s = frame[cols[0]].astype(str)
    for c in cols[1:]:
        s = s + KEY_SEP + frame[c].astype(str)
    return s


def _sum_count(frame, cols):
    g = frame['y'].astype(float).groupby(_keystr(frame, cols).to_numpy()).agg(['sum', 'count'])
    return {str(k): [float(s), int(n)] for k, s, n in zip(g.index, g['sum'], g['count'])}


def fit_baseline(spec, train, cutoff=None):
    """spec from config: zero | levels (coarse -> fine group means over the Train global mean) [+ smooth k] [+ recent window frozen at cutoff]."""
    if not len(train):
        raise ValueError(f"{spec['id']}: empty Train")
    st = {'id': spec['id'], 'spec': spec, 'n_train': int(len(train)), 'global_mean': float(train['y'].astype(float).mean()),
          'levels': [], 'smooth': None, 'recent': None, 'frozen_at': None}
    if spec.get('zero'):
        return st
    st['levels'] = [{'cols': list(c), 'table': _sum_count(train, list(c))} for c in spec.get('levels', [])]
    if spec.get('smooth'):
        sm = spec['smooth']
        st['smooth'] = {'cols': list(sm['cols']), 'k': float(sm['k']), 'table': _sum_count(train, list(sm['cols']))}
    if spec.get('recent'):
        rc = spec['recent']
        if cutoff is None:
            raise ValueError(f"{spec['id']}: a recent-history baseline must be frozen at an explicit fold-start cutoff")
        cutoff = pd.Timestamp(cutoff)
        if (train['end_ts'] > cutoff).any():
            raise ValueError(f"{spec['id']}: Train contains cells ending after the frozen cutoff")
        lo = cutoff - pd.Timedelta(days=int(rc['days']))
        w = train[train['end_ts'] > lo]
        st['recent'] = {'cols': list(rc['cols']), 'days': int(rc['days']), 'window_end_exclusive_lo': lo.isoformat(), 'n_window_cells': int(len(w)),
                        'table': _sum_count(w, list(rc['cols']))}
        st['frozen_at'] = cutoff.isoformat()
    return st


def _mean_map(frame, cols, table):
    k = _keystr(frame, cols)
    return k.map({key: s / n for key, (s, n) in table.items() if n > 0}).to_numpy(dtype=float)


def predict_baseline(st, frame):
    """Uses only the fitted state (never the evaluation labels); unseen keys fall back to the coarser level / global mean."""
    n = len(frame)
    if st['spec'].get('zero'):
        return np.zeros(n)
    pred = np.full(n, st['global_mean'], dtype=float)
    for lv in st['levels']:
        m = _mean_map(frame, lv['cols'], lv['table'])
        pred = np.where(np.isnan(m), pred, m)
    if st['smooth']:
        sm = st['smooth']
        k = _keystr(frame, sm['cols'])
        s = k.map({kk: v[0] for kk, v in sm['table'].items()}).fillna(0.0).to_numpy(float)
        c = k.map({kk: v[1] for kk, v in sm['table'].items()}).fillna(0.0).to_numpy(float)
        pred = (s + sm['k'] * pred) / (c + sm['k'])
    if st['recent']:
        m = _mean_map(frame, st['recent']['cols'], st['recent']['table'])
        pred = np.where(np.isnan(m), pred, m)
    return pred


# ---------- train-only preprocessing + candidate families ----------
def make_preprocessor(kind, cat_cols, num_cols):
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
    # add_indicator=False: missingness / source-coverage flags are NOT approved predictors; all-missing Train columns are kept (0 fallback)
    if kind == 'onehot_impute_scale':
        cat = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
        num = Pipeline([('impute', SimpleImputer(strategy='median', add_indicator=False, keep_empty_features=True)), ('scale', StandardScaler())])
    elif kind == 'onehot_impute':
        cat = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
        num = SimpleImputer(strategy='median', add_indicator=False, keep_empty_features=True)
    elif kind == 'ordinal_native':
        cat = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=np.nan, encoded_missing_value=np.nan)
        num = 'passthrough'
    else:
        raise ValueError(f'unknown preprocessor {kind!r}')
    tr = ([('cat', cat, list(cat_cols))] if cat_cols else []) + ([('num', num, list(num_cols))] if num_cols else [])
    return ColumnTransformer(tr, remainder='drop', verbose_feature_names_out=True)


def make_model(cfg, n_cat=0, n_out=None):
    from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
    from sklearn.linear_model import PoissonRegressor
    fam, p = cfg['family'], dict(cfg['params'])
    if fam == 'poisson':
        return PoissonRegressor(**p)
    if fam == 'rf':
        return RandomForestRegressor(**p)
    if fam == 'hgb':
        mask = [True] * n_cat + [False] * (n_out - n_cat)
        return HistGradientBoostingRegressor(categorical_features=mask if n_cat else None, **p)
    raise ValueError(f'unknown family {fam!r}')


def make_pipeline(cfg, cat_cols, num_cols):
    from sklearn.pipeline import Pipeline
    prep = make_preprocessor(cfg['preprocessor'], cat_cols, num_cols)
    return Pipeline([('prep', prep), ('model', make_model(cfg, len(cat_cols), len(cat_cols) + len(num_cols)))])


def validate_predictions(mu, label=''):
    mu = np.asarray(mu, dtype=float)
    if not np.isfinite(mu).all():
        raise ValueError(f'{label}: non-finite predictions')
    if (mu < 0).any():
        raise ValueError(f'{label}: negative predictions')
    return mu


def logged_call(fn, threads):
    """Run fn() with bounded thread pools while recording EVERY warning (convergence warnings are counted, never silenced)."""
    from sklearn.exceptions import ConvergenceWarning
    from threadpoolctl import threadpool_limits
    t0 = time.perf_counter()
    with warnings.catch_warnings(record=True) as w, threadpool_limits(limits=threads):
        warnings.simplefilter('always')
        out = fn()
    msgs = [f'{x.category.__name__}: {str(x.message)[:300]}' for x in w]
    return out, {'seconds': round(time.perf_counter() - t0, 3), 'n_warnings': len(w), 'warnings': msgs[:10],
                 'n_convergence_warnings': int(sum(issubclass(x.category, ConvergenceWarning) for x in w))}


def unknown_levels(train_X, eval_X, cat_cols):
    return {c: int((~eval_X[c].isin(set(train_X[c].dropna()))).sum()) for c in cat_cols}


def fit_prep_group(kind, cfgs, Xtr, ytr, Xevs, cat_cols, num_cols, threads):
    """Fit ONE preprocessor on this fold's Train only, transform Train/evaluation once, fit every configuration that shares it.
    Returns {config_id: {'preds': [array per eval] | None, 'log': {...}, 'model': fitted | None}}, fitted preprocessor."""
    out = {}
    try:
        prep = make_preprocessor(kind, cat_cols, num_cols)
        (Ttr, Tevs), plog = logged_call(lambda: (prep.fit_transform(Xtr), [prep.transform(x) for x in Xevs]), threads)
    except Exception as e:  # recorded; every config of this group fails (no fallback model)
        for c in cfgs:
            out[c['id']] = {'preds': None, 'model': None, 'log': {'status': 'failed', 'error': f'preprocessor: {type(e).__name__}: {e}'[:500]}}
        return out, None
    n_out = Ttr.shape[1]
    for c in cfgs:
        try:
            model = make_model(c, len(cat_cols), n_out)
            _, log = logged_call(lambda: model.fit(Ttr, ytr), threads)
            preds = [validate_predictions(model.predict(T), c['id']) for T in Tevs]
            log.update(status='ok', error=None, n_iter=None if getattr(model, 'n_iter_', None) is None else int(np.max(model.n_iter_)),
                       n_features_in=int(n_out), prep_seconds=plog['seconds'], prep_warnings=plog['warnings'])
            out[c['id']] = {'preds': preds, 'model': model, 'log': log}
        except Exception as e:
            out[c['id']] = {'preds': None, 'model': None, 'log': {'status': 'failed', 'error': f'{type(e).__name__}: {e}'[:500]}}
    return out, prep


def method_id(config_id, variant):
    return f'{config_id}__{variant}'


def baseline_method_id(bid):
    return f'baseline_{bid}'


def run_unit_oof(frame, folds, variant_cols, cat_cols, model_cfgs, baseline_specs, threads=2, log=print, keys=('province_key', 'target_date')):
    """folds: list of dicts {fold, issue, train_mask, eval_mask}. Every method predicts EVERY evaluation row of every fold (identical keys);
    feature NaN never drops rows. Baselines are fitted once per fold (they do not depend on the feature variant). A failed fit leaves NaN and
    marks the method incomplete; nothing is replaced by another model. Returns oof frame, preds {method: array}, fit_log rows, baseline states."""
    parts, offs = [], []
    o = 0
    for f in folds:
        ev = frame.loc[f['eval_mask'], list(dict.fromkeys(list(keys) + ['target_date', 'y']))].copy()
        ev.insert(0, 'fold', f['fold'])
        parts.append(ev)
        offs.append((o, o + len(ev)))
        o += len(ev)
    oof = pd.concat(parts, ignore_index=True)
    n = len(oof)
    preds, fit_log, bstates = {}, [], {}
    by_prep = {}
    for c in model_cfgs:
        by_prep.setdefault(c['preprocessor'], []).append(c)
    for f, (a, b) in zip(folds, offs):
        tr, ev = frame.loc[f['train_mask']], frame.loc[f['eval_mask']]
        for spec in baseline_specs:
            mid = baseline_method_id(spec['id'])
            t0 = time.perf_counter()
            st = fit_baseline(spec, tr, cutoff=f['issue'])
            p = validate_predictions(predict_baseline(st, ev), mid)
            preds.setdefault(mid, np.full(n, np.nan))[a:b] = p
            bstates[(f['fold'], spec['id'])] = st
            fit_log.append({'fold': f['fold'], 'method_id': mid, 'family': 'baseline', 'config_id': spec['id'], 'variant': '(none)', 'status': 'ok',
                            'seconds': round(time.perf_counter() - t0, 3), 'n_train': len(tr), 'n_eval': len(ev), 'n_warnings': 0,
                            'n_convergence_warnings': 0, 'n_iter': None, 'warnings': '', 'error': None, 'unknown_levels': ''})
        ytr = tr['y'].to_numpy(float)
        for v, cols in variant_cols.items():
            cc = [c for c in cols if c in cat_cols]
            nc = [c for c in cols if c not in cat_cols]
            Xtr, Xev = tr[cc + nc], ev[cc + nc]
            unk = unknown_levels(Xtr, Xev, cc)
            for kind, cfgs in by_prep.items():
                res, _ = fit_prep_group(kind, cfgs, Xtr, ytr, [Xev], cc, nc, threads)
                for c in cfgs:
                    r, mid = res[c['id']], method_id(c['id'], v)
                    arr = preds.setdefault(mid, np.full(n, np.nan))
                    if r['preds'] is not None:
                        arr[a:b] = r['preds'][0]
                    lg = r['log']
                    fit_log.append({'fold': f['fold'], 'method_id': mid, 'family': c['family'], 'config_id': c['id'], 'variant': v,
                                    'status': lg['status'], 'seconds': lg.get('seconds'), 'n_train': len(tr), 'n_eval': len(ev),
                                    'n_warnings': lg.get('n_warnings'), 'n_convergence_warnings': lg.get('n_convergence_warnings'),
                                    'n_iter': lg.get('n_iter'), 'warnings': ' | '.join(lg.get('warnings', []) + lg.get('prep_warnings', [])),
                                    'error': lg.get('error'), 'unknown_levels': json.dumps(unk, ensure_ascii=False)})
                    log(f"  {f['fold']} {mid}: {lg['status']} {lg.get('seconds')}s warnings={lg.get('n_warnings')}")
    for mid, arr in preds.items():
        oof[mid] = arr
    return oof, preds, fit_log, bstates


# ---------- metrics ----------
def poisson_deviance_rows(y, mu, floor):
    """Unit Poisson deviance per row; mu floored at `floor` ONLY here (count returned). Negative/non-finite predictions raise."""
    from scipy.special import xlogy
    y = np.asarray(y, dtype=float)
    mu = validate_predictions(mu, 'deviance')
    n_floor = int((mu < floor).sum())
    m = np.maximum(mu, floor)
    return 2.0 * (xlogy(y, y) - xlogy(y, m) - (y - m)), n_floor


def metric_summary(y, mu, floor, deviance=True):
    y, mu = np.asarray(y, dtype=float), validate_predictions(mu, 'metrics')
    e = mu - y
    sy = float(y.sum())
    out = {'n': int(len(y)), 'sum_y': sy, 'sum_pred': float(mu.sum()), 'mae': float(np.abs(e).mean()), 'rmse': float(np.sqrt((e ** 2).mean())),
           'wape': None if sy == 0 else float(np.abs(e).sum() / sy), 'bias': float(e.mean()), 'mean_poisson_deviance': None, 'n_deviance_floored': None}
    if deviance:
        d, nf = poisson_deviance_rows(y, mu, floor)
        out.update(mean_poisson_deviance=float(d.mean()), n_deviance_floored=nf)
    return out


def is_zero_method(mid):
    return mid == baseline_method_id('zero')


def method_meta(mid, model_cfgs):
    if mid.startswith('baseline_'):
        return {'family': 'baseline', 'config_id': mid[len('baseline_'):], 'variant': '(none)'}
    cid, v = mid.split('__', 1)
    fam = {c['id']: c['family'] for c in model_cfgs}[cid]
    return {'family': fam, 'config_id': cid, 'variant': v}


def oof_metric_tables(oof, methods, model_cfgs, floor, failed):
    """Pooled and per-fold metrics for every method on the identical OOF keys; incomplete (failed) methods get status 'failed', no metrics."""
    pooled, byfold = [], []
    y = oof['y'].to_numpy(float)
    for mid in methods:
        meta = method_meta(mid, model_cfgs)
        mu = oof[mid].to_numpy(float)
        if mid in failed or np.isnan(mu).any():
            pooled.append(dict(method_id=mid, **meta, status='failed'))
            continue
        pooled.append(dict(method_id=mid, **meta, status='ok', **metric_summary(y, mu, floor, deviance=not is_zero_method(mid))))
        for fold, g in oof.groupby('fold', sort=False):
            byfold.append(dict(method_id=mid, **meta, fold=fold, **metric_summary(g['y'], g[mid], floor, deviance=not is_zero_method(mid))))
    return pd.DataFrame(pooled), pd.DataFrame(byfold)


# ---------- selection ----------
def pick_best(rows, family_order=()):
    """Lowest pooled OOF mean deviance among status ok rows; exact ties -> family_order then method_id."""
    ok = [r for r in rows if r.get('status') == 'ok' and r.get('mean_poisson_deviance') is not None and math.isfinite(r['mean_poisson_deviance'])]
    if not ok:
        return None
    rank = {f: i for i, f in enumerate(family_order)}
    return min(ok, key=lambda r: (r['mean_poisson_deviance'], rank.get(r['family'], len(rank)), r['method_id']))


def select_unit(pooled, family_order):
    rows = pooled.to_dict('records')
    fam_best = {f: pick_best([r for r in rows if r['family'] == f]) for f in family_order}
    cand = pick_best([r for r in fam_best.values() if r is not None], family_order)
    base = pick_best([r for r in rows if r['family'] == 'baseline' and not is_zero_method(r['method_id'])])
    failed = sorted(r['method_id'] for r in rows if r['status'] != 'ok')
    return {'family_best': {f: (None if r is None else r['method_id']) for f, r in fam_best.items()},
            'best_candidate': None if cand is None else cand['method_id'], 'strongest_baseline': None if base is None else base['method_id'],
            'failed_methods': failed}


def calendar_blocks(dates, anchor, block_days):
    d = pd.to_datetime(pd.Series(dates), format='%Y-%m-%d')
    return ((d - pd.Timestamp(anchor)).dt.days // int(block_days)).to_numpy()


def block_bootstrap(diff, blocks, n_resamples, seed, ci=0.95):
    """Paired calendar-block bootstrap of the row-weighted mean difference: whole blocks (all rows of those days) are resampled together."""
    diff = np.asarray(diff, dtype=float)
    ub, inv = np.unique(np.asarray(blocks), return_inverse=True)
    sums, cnts = np.bincount(inv, weights=diff), np.bincount(inv).astype(float)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(ub), size=(int(n_resamples), len(ub)))
    stats = sums[idx].sum(axis=1) / cnts[idx].sum(axis=1)
    a = (1 - ci) / 2
    return {'point': float(diff.mean()), 'lower': float(np.quantile(stats, a)), 'upper': float(np.quantile(stats, 1 - a)), 'n_blocks': int(len(ub)),
            'n_resamples': int(n_resamples), 'seed': int(seed), 'ci': ci, 'stats': stats}


def stability_gate(cand, base, cand_folds, base_folds, boot, g):
    """cand/base: pooled metric dicts; *_folds: {fold: mean deviance}. All four checks must pass."""
    rel = (base['mean_poisson_deviance'] - cand['mean_poisson_deviance']) / base['mean_poisson_deviance']
    wins = sum(cand_folds[f] < base_folds[f] for f in base_folds)
    checks = {'relative_deviance_reduction': {'value': rel, 'threshold': g['min_relative_deviance_reduction'], 'pass': bool(rel >= g['min_relative_deviance_reduction'])},
              'mae_ratio': {'value': cand['mae'] / base['mae'] if base['mae'] > 0 else None, 'threshold': g['max_mae_ratio'],
                            'pass': bool(cand['mae'] <= g['max_mae_ratio'] * base['mae'])},
              'fold_wins': {'value': int(wins), 'of': len(base_folds), 'threshold': g['min_fold_wins'], 'pass': bool(wins >= g['min_fold_wins'])},
              'bootstrap_upper_ci_lt_0': {'value': boot['upper'], 'lower': boot['lower'], 'point': boot['point'], 'threshold': 0.0, 'pass': bool(boot['upper'] < 0)}}
    return {'passed': all(c['pass'] for c in checks.values()), 'checks': checks}


# ---------- selection lock ----------
SELECTION_LOCK_SCHEMA = 'roadaware-v2-selection-lock/1'


def write_selection_lock(path, payload):
    if Path(path).exists():
        raise LockError(f'selection lock {path} already exists in this run; refusing to overwrite')
    write_json(path, dict(payload, schema=SELECTION_LOCK_SCHEMA))
    return sha256_file(path)


def verify_selection_lock(path, expected_sha, actual_hashes, base_dir):
    """The lock file bytes, every hash it records for files (relative to base_dir) and the given actual input hashes must be unchanged."""
    p = Path(path)
    if not p.is_file():
        raise LockError(f'selection lock {p} missing')
    if sha256_file(p) != expected_sha:
        raise LockError('selection lock bytes changed after it was written')
    lock = read_json_strict(p)
    if lock.get('schema') != SELECTION_LOCK_SCHEMA:
        raise LockError('selection lock schema mismatch')
    for e in lock['files']:
        got = sha256_file(Path(base_dir) / e['path'])
        if got != e['sha256']:
            raise LockError(f"{e['path']} changed since the selection lock ({got[:12]} != {e['sha256'][:12]})")
    for k, v in actual_hashes.items():
        if lock['hashes'].get(k) != v:
            raise LockError(f'{k} differs from the selection lock')
    return lock


# ---------- bundles / reload ----------
def save_model_bundle(path, pipeline, meta):
    import joblib
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump({'pipeline': pipeline, 'meta': meta}, path, compress=3)
    return sha256_file(path)


def load_model_bundle(path, expected_sha):
    import joblib
    verify_file(path, expected_sha, 'model bundle')
    return joblib.load(path)


def reload_consistency(mem, reloaded, tol):
    mem, reloaded = np.asarray(mem, dtype=float), np.asarray(reloaded, dtype=float)
    if mem.shape != reloaded.shape:
        return {'n': int(mem.size), 'max_abs_diff': None, 'pass': False}
    d = float(np.max(np.abs(mem - reloaded))) if mem.size else 0.0
    return {'n': int(mem.size), 'max_abs_diff': d, 'pass': bool(d <= tol)}

## A. Roots, experiment lock และการตรวจอินพุต

ตรวจ manifest ของ 10, lock ของ manager (protocol/config/manifest/data hashes, run ids), allowlist feature และ guard ชื่อต้องห้าม ก่อนโหลด panel

In [ ]:
# A. Roots, config, manager experiment lock and strict input verification (no fit happens unless every check passes)
import sklearn, scipy, joblib, threadpoolctl
T_RUN0 = time.perf_counter()
BACKEND = find_backend()
REPO = BACKEND.parent
CFG_PATH = BACKEND / 'config' / 'model_experiment_v2.json'
CFG = read_json_strict(CFG_PATH)
IC = CFG['inputs']
NB11 = BACKEND / 'notebooks' / '11_v2_model_experiments.ipynb'
OUT = env_path('ROADAWARE_V2_OUTPUT_ROOT', BACKEND)
PREP_DIR = env_path('ROADAWARE_V2_PREPARATION_DIR', BACKEND / IC['preparation_report_subdir'])   # read-only accepted 10 reports
PREP_DATA = env_path('ROADAWARE_V2_DATA_ROOT', BACKEND / IC['preparation_data_subdir'])          # read-only accepted 10 matrices
SRC_DIR = env_path('ROADAWARE_V2_SOURCES_DIR', BACKEND / IC['sources_report_subdir'])            # read-only, run-id cross-check only
MODEL_DATA = env_path('ROADAWARE_V2_MODEL_DATA_ROOT', OUT / CFG['outputs']['model_data_subdir'])  # gitignored row-level outputs
REP = OUT / CFG['outputs']['report_subdir']                                                       # tracked aggregate evidence
LOCK_PATH = env_path('ROADAWARE_V2_EXPERIMENT_LOCK', REPO / IC['experiment_lock'])
PROTOCOL_PATH = REPO / IC['protocol_draft']
FCFG_PATH = BACKEND / IC['feature_config']
V1 = env_path('ROADAWARE_V1_DATA_ROOT', BACKEND)
V1_PROTECTED = [V1 / 'data' / r for r in ('raw', 'interim', 'processed', 'model')] + [V1 / 'reports' / r for r in ('data', 'models')]

PM_PATH = PREP_DIR / IC['preparation_manifest']
if not PM_PATH.is_file():
    raise DependencyError(f'accepted notebook 10 manifest not found: {PM_PATH}')
PM = read_json_strict(PM_PATH)
verify_preparation_manifest(PM, IC)
PREP_FILES = [PREP_DATA / o['path'] for o in PM['data_outputs']]
check_write_roots(REP, MODEL_DATA, PREP_DIR, PREP_DATA, [('09 sources dir', SRC_DIR)] + [('protected v1', v) for v in V1_PROTECTED], PREP_FILES)
LOCK = verify_experiment_lock(LOCK_PATH, PROTOCOL_PATH, CFG_PATH, PM_PATH, PM, IC['experiment_lock_schema'], IC['experiment_lock_fields'])
SRC_RUN_CHECK = check_source_run_id(LOCK, PM, SRC_DIR)
if sha256_file(FCFG_PATH) != PM['config_sha256']:
    raise DependencyError('config/feature_pipeline_v2.json differs from the config hash recorded by the accepted 10 run')
FCFG = read_json_strict(FCFG_PATH)
IFACE = PM['next_stage_interface']
PM_SHA = {o['path']: o['sha256'] for o in PM['data_outputs']}
INPUTS = [{'role': 'experiment_lock', 'path': str(LOCK_PATH), 'sha256': sha256_file(LOCK_PATH)},
          {'role': 'protocol_draft', 'path': str(PROTOCOL_PATH), 'sha256': LOCK['protocol_sha256']},
          {'role': 'model_config', 'path': str(CFG_PATH), 'sha256': LOCK['model_config_sha256']},
          {'role': 'preparation_manifest', 'path': str(PM_PATH), 'sha256': LOCK['preparation_manifest_sha256']},
          {'role': 'feature_config', 'path': str(FCFG_PATH), 'sha256': PM['config_sha256']}]
for unit in CFG['units']:
    for part in ('panel', 'features'):
        name = IFACE[unit][part]
        INPUTS.append({'role': f'{unit}_{part}', 'path': str(PREP_DATA / name), 'sha256': verify_file(PREP_DATA / name, PM_SHA.get(name), f'10 {unit} {part}')})

# approved feature guard: the 10 allowlist for each unit, the forbidden names/substrings and the never-model-input list
OFFSET = CFG['time']['utc_offset']
FRAMES, VARIANTS, CATS, UNIT_COLS = {}, {}, {}, {}
for unit in CFG['units']:
    it = IFACE[unit]
    approved = [f['name'] for f in FCFG['features'] if unit in f['panels']]
    if list(it['feature_columns']) != approved:
        raise DependencyError(f'{unit}: 10 interface feature_columns differ from the approved feature_pipeline_v2 allowlist')
    validate_feature_columns(it['feature_columns'], approved, FCFG['forbidden_feature_names'], FCFG['forbidden_feature_substrings'],
                             never=IFACE['never_model_inputs'])
    UNIT_COLS[unit] = list(it['feature_columns'])
    CATS[unit] = [c for c in IFACE['categorical'] if c in UNIT_COLS[unit]]
    VARIANTS[unit] = variant_columns(UNIT_COLS[unit], IFACE['feature_groups'], CFG['feature_variants'], CFG['unit_extra_groups'][unit])
    if sorted(VARIANTS[unit]['full']) != sorted(UNIT_COLS[unit]):
        raise ValueError(f'{unit}: variant full must use every approved column')
    FRAMES[unit] = load_unit_frame(PREP_DATA / it['panel'], PREP_DATA / it['features'], it, CATS[unit], OFFSET)
    fr = FRAMES[unit]
    print(f"{unit}: rows={len(fr)} eligible={int(fr['eligible'].sum())} splits={fr['split'].value_counts().to_dict()} "
          f"categorical={CATS[unit]} variants={ {v: len(c) for v, c in VARIANTS[unit].items()} }")

VERSIONS = {'python': sys.version.split()[0], 'numpy': np.__version__, 'pandas': pd.__version__, 'scikit-learn': sklearn.__version__,
            'scipy': scipy.__version__, 'joblib': joblib.__version__, 'threadpoolctl': threadpoolctl.__version__}
THREADS = int(CFG['compute']['threads'])
assert all(c['params'].get('n_jobs', THREADS) == THREADS for c in CFG['model_configs'])
_old = existing_results(REP, MODEL_DATA)
if _old:  # fail fast BEFORE the long OOF fits; earlier locks/results are preserved, never overwritten
    raise RuntimeError(f'output roots already hold a notebook 11 run ({_old[:4]}). Use fresh roots, e.g.\n'
                       '  .venv/bin/python backend/scripts/run_v2_notebooks.py --stage model --output-dir backend/rerun-model-<UTC-stamp>\n'
                       'or in VS Code set ROADAWARE_V2_OUTPUT_ROOT=<fresh dir> (and ROADAWARE_V2_MODEL_DATA_ROOT if overridden) before Run All.')
MODEL_DATA.mkdir(parents=True, exist_ok=True)
REP.mkdir(parents=True, exist_ok=True)
RUN = {'started_utc': utc_now(), 'timings': {}, 'source_run_check': SRC_RUN_CHECK}
pd.set_option('display.width', 200)
print(f'backend={BACKEND}\nprep_dir={PREP_DIR} (read-only)\nprep_data={PREP_DATA} (read-only)\nmodel_data={MODEL_DATA}\nreports={REP}\n'
      f'lock={LOCK_PATH}\nsource_run_check={SRC_RUN_CHECK}\nversions={VERSIONS}')

## B. Cohort และ fold ตามเวลา issue

ทุกวิธีใช้ keys ชุดเดียวกันต่อหน่วย; ตรวจ train end <= issue, keys ไม่ซ้อน, ไม่มี fit ก่อนผ่านการตรวจ; final cutoff 2023-12-31 18:00 +07

In [ ]:
# B. Cohorts and issue-time folds per unit (checked BEFORE any fit): identical eligible keys for every method, disjoint Train/evaluation keys
T = CFG['time']
FOLDS, FINAL, BOUNDS, COHORT = {}, {}, [], []
FINAL_CUTOFF = pd.Timestamp(CFG['final_fit']['cutoff_local'])
if FINAL_CUTOFF != issue_time_for(CFG['final_fit']['first_diagnostic_date'], T['issue_hour_local'], T['issue_days_before_target'], OFFSET):
    raise ValueError('final-fit cutoff must equal the issue time of the first diagnostic date')
for unit in CFG['units']:
    fr, keys = FRAMES[unit], list(IFACE[unit]['keys'])
    check_issue_times(fr, T['issue_hour_local'], T['issue_days_before_target'], OFFSET)
    FOLDS[unit] = []
    for f in CFG['folds']:
        issue = issue_time_for(f['eval_start'], T['issue_hour_local'], T['issue_days_before_target'], OFFSET)
        tm, em = fit_mask(fr, issue, T['train_split']), eval_mask(fr, f['eval_start'], f['eval_end'], T['train_split'])
        chk = check_fold(fr, tm, em, issue, keys, f['name'])
        before = fr['split'].eq(T['train_split']).to_numpy() & (fr['target_date'] < f['eval_start']).to_numpy()
        allm = (fr['target_date'] >= f['eval_start']).to_numpy() & (fr['target_date'] <= f['eval_end']).to_numpy()
        chk.update(unit=unit, eval_start=f['eval_start'], eval_end=f['eval_end'],
                   n_eval_ineligible_excluded=int((allm & ~fr['eligible'].to_numpy()).sum()),
                   n_train_ineligible_excluded=int((before & ~fr['eligible'].to_numpy() & (fr['end_ts'] <= issue).to_numpy()).sum()),
                   n_gap_cells_after_issue_not_fitted=int((before & fr['eligible'].to_numpy() & (fr['end_ts'] > issue).to_numpy()).sum()))
        BOUNDS.append(chk)
        FOLDS[unit].append({'fold': f['name'], 'issue': issue, 'train_mask': tm, 'eval_mask': em})
    # pooled OOF evaluation keys: every fold's evaluation set, disjoint across folds
    ems = [f['eval_mask'] for f in FOLDS[unit]]
    assert not np.any(np.sum(ems, axis=0) > 1), 'evaluation folds overlap'
    # final diagnostic fit: original Train only, every target ended by the first 2024 issue time
    ftm = fit_mask(fr, FINAL_CUTOFF, T['train_split'])
    tr_el = fr['eligible'].to_numpy() & fr['split'].eq(T['train_split']).to_numpy()
    diag = {s['name']: fr['eligible'].to_numpy() & fr['split'].eq(s['name']).to_numpy() for s in CFG['final_fit']['diagnostic_splits']}
    for name, dm in diag.items():
        if not dm.any():
            raise ValueError(f'{unit}: no eligible {name} rows')
        check_fold(fr, ftm, dm, FINAL_CUTOFF, keys, f'final->{name}')
    FINAL[unit] = {'train_mask': ftm, 'diag_masks': diag, 'n_train': int(ftm.sum()),
                   'n_excluded_after_cutoff': int((tr_el & ~ftm).sum()), 'cutoff': FINAL_CUTOFF.isoformat(),
                   'max_train_target_end': fr.loc[ftm, 'end_ts'].max().isoformat()}
    BOUNDS.append({'unit': unit, 'fold': 'final_fit', 'issue_time_local': FINAL_CUTOFF.isoformat(), 'n_train': int(ftm.sum()),
                   'train_first_target_date': fr.loc[ftm, 'target_date'].min(), 'train_last_target_date': fr.loc[ftm, 'target_date'].max(),
                   'train_max_target_end_local': FINAL[unit]['max_train_target_end'], 'n_gap_cells_after_issue_not_fitted': FINAL[unit]['n_excluded_after_cutoff'],
                   'train_y_sum': float(fr.loc[ftm, 'y'].sum()), 'disjoint_keys': True, 'train_end_le_issue': True})
    for name, dm in diag.items():
        BOUNDS.append({'unit': unit, 'fold': f'diagnostic:{name}', 'issue_time_local': FINAL_CUTOFF.isoformat(), 'n_eval': int(dm.sum()),
                       'eval_first_target_date': fr.loc[dm, 'target_date'].min(), 'eval_last_target_date': fr.loc[dm, 'target_date'].max(),
                       'eval_y_sum': float(fr.loc[dm, 'y'].sum()),
                       'n_eval_ineligible_excluded': int((fr['split'].eq(name).to_numpy() & ~fr['eligible'].to_numpy()).sum())})
    # feature missingness by split (eligible cells; missing values stay in every cohort, they never drop rows)
    for sp, g in fr[fr['eligible']].groupby('split'):
        for c in UNIT_COLS[unit]:
            if c in keys or c in CATS[unit]:
                continue
            COHORT.append({'unit': unit, 'split': sp, 'feature': c, 'n_eligible': int(len(g)), 'non_null_share': float(g[c].notna().mean())})
BOUNDS = pd.DataFrame(BOUNDS)
COHORT = pd.DataFrame(COHORT)
write_csv(REP / 'fold_boundaries.csv', BOUNDS)
write_csv(REP / 'feature_missingness_by_split.csv', COHORT)
RUN['final_fit'] = {u: {k: v for k, v in FINAL[u].items() if not k.endswith('mask') and k != 'diag_masks'} for u in FINAL}
print(BOUNDS.to_string(index=False))
print(RUN['final_fit'])

## C. OOF backtest

baseline fit ครั้งเดียวต่อ fold (ไม่ขึ้นกับ variant); preprocessor fit ต่อ fold/variant และแชร์เฉพาะระหว่าง Poisson สอง alpha ใน fold เดียวกัน; warning/เวลา/ความล้มเหลวถูกบันทึกทุกครั้ง (ไม่มี fallback model)

In [ ]:
# C1. Out-of-fold backtest, DAILY unit: 6 baselines + 4 configurations x 5 variants on the four expanding folds (preprocessing fitted per fold Train)
OOF, FITLOG, BSTATES = globals().get('OOF', {}), globals().get('FITLOG', {}), globals().get('BSTATES', {})
_t0 = time.perf_counter()
OOF['daily'], _, FITLOG['daily'], BSTATES['daily'] = run_unit_oof(FRAMES['daily'], FOLDS['daily'], VARIANTS['daily'], CATS['daily'], CFG['model_configs'],
                                                                CFG['baselines']['daily'], threads=THREADS, keys=IFACE['daily']['keys'])
RUN['timings']['oof_daily_seconds'] = round(time.perf_counter() - _t0, 1)
print('daily OOF rows', len(OOF['daily']), 'methods', len([c for c in OOF['daily'] if c.startswith(('baseline_', 'poisson', 'rf', 'hgb'))]),
      'seconds', RUN['timings']['oof_daily_seconds'])

In [ ]:
# C2. Out-of-fold backtest, 6 h unit (separate experiment; never compared with daily errors as if the targets matched)
_t0 = time.perf_counter()
OOF['6h'], _, FITLOG['6h'], BSTATES['6h'] = run_unit_oof(FRAMES['6h'], FOLDS['6h'], VARIANTS['6h'], CATS['6h'], CFG['model_configs'],
                                                       CFG['baselines']['6h'], threads=THREADS, keys=IFACE['6h']['keys'])
RUN['timings']['oof_6h_seconds'] = round(time.perf_counter() - _t0, 1)
# save exact OOF keys/predictions locally (gitignored) and freeze their hashes
OOF_FILES = {}
for unit in CFG['units']:
    p = MODEL_DATA / f'oof_predictions_{unit}.csv.gz'
    write_csv_gz(p, OOF[unit])
    OOF_FILES[unit] = {'path': p.name, 'sha256': sha256_file(p), 'rows': int(len(OOF[unit]))}
FITLOG_DF = pd.concat([pd.DataFrame(FITLOG[u]).assign(unit=u) for u in CFG['units']], ignore_index=True)
write_csv(REP / 'fit_log.csv', FITLOG_DF)
print('6h OOF rows', len(OOF['6h']), 'seconds', RUN['timings']['oof_6h_seconds'])
print(FITLOG_DF.groupby(['unit', 'family', 'status']).agg(fits=('method_id', 'size'), seconds=('seconds', 'sum'),
                                                          convergence_warnings=('n_convergence_warnings', 'sum')).to_string())

## D/E. Metric, ablation, การเลือก, gate, bootstrap และ selection lock

เลือกแยกต่อหน่วยจาก pooled OOF deviance; gate: ลด deviance >= 1%, MAE <= 1.02x, ชนะ >= 3/4 fold, upper CI95 ของ paired 7-day block bootstrap < 0 (สำรวจ ไม่ปรับ multiple comparison)

In [ ]:
# D/E. Pooled OOF metrics, ablation deltas on identical keys, per-unit selection, stability gate, exploratory bootstrap and the SELECTION LOCK
FLOOR = float(CFG['metrics']['deviance_floor'])
G = CFG['gate']
POOLED, BYFOLD, ABL, SEL = [], [], [], {}
for unit in CFG['units']:
    oof = OOF[unit]
    methods = [c for c in oof.columns if c.startswith('baseline_') or '__' in c]
    failed = set(FITLOG_DF.loc[FITLOG_DF['unit'].eq(unit) & FITLOG_DF['status'].ne('ok'), 'method_id'])
    pooled, byfold = oof_metric_tables(oof, methods, CFG['model_configs'], FLOOR, failed)
    POOLED.append(pooled.assign(unit=unit))
    BYFOLD.append(byfold.assign(unit=unit))
    fd = {m: g.set_index('fold')['mean_poisson_deviance'].to_dict() for m, g in byfold.groupby('method_id')} if len(byfold) else {}
    pm = pooled.set_index('method_id')
    # ablation: benefit AND harm of each fixed feature group, same configuration, identical OOF keys
    for c in CFG['model_configs']:
        for v_new, v_ref in CFG['ablation_comparisons']:
            a, b = method_id(c['id'], v_new), method_id(c['id'], v_ref)
            if pm.loc[a, 'status'] != 'ok' or pm.loc[b, 'status'] != 'ok':
                ABL.append({'unit': unit, 'config_id': c['id'], 'family': c['family'], 'variant': v_new, 'reference': v_ref, 'status': 'failed'})
                continue
            da, db = pm.loc[a, 'mean_poisson_deviance'], pm.loc[b, 'mean_poisson_deviance']
            delta = da - db
            ABL.append({'unit': unit, 'config_id': c['id'], 'family': c['family'], 'variant': v_new, 'reference': v_ref, 'status': 'ok',
                        'deviance_variant': da, 'deviance_reference': db, 'delta_deviance': delta, 'relative_change': delta / db,
                        'delta_mae': pm.loc[a, 'mae'] - pm.loc[b, 'mae'], 'folds_improved': int(sum(fd[a][f] < fd[b][f] for f in fd[b])),
                        'effect': 'benefit' if delta < 0 else ('harm' if delta > 0 else 'no_change'), 'n_rows': int(pm.loc[a, 'n'])})
    s = select_unit(pooled, CFG['family_order'])
    if s['best_candidate'] is None or s['strongest_baseline'] is None:
        s.update(gate={'passed': False, 'checks': {}, 'reason': 'no complete candidate or baseline'}, bootstrap=None)
    else:
        cand, base = pm.loc[s['best_candidate']].to_dict(), pm.loc[s['strongest_baseline']].to_dict()
        dc, _ = poisson_deviance_rows(oof['y'], oof[s['best_candidate']], FLOOR)
        db, _ = poisson_deviance_rows(oof['y'], oof[s['strongest_baseline']], FLOOR)
        blocks = calendar_blocks(oof['target_date'], CFG['folds'][0]['eval_start'], G['bootstrap']['block_days'])
        boot = block_bootstrap(dc - db, blocks, G['bootstrap']['n_resamples'], G['bootstrap']['seed'], G['bootstrap']['ci'])
        boot.pop('stats')
        s['bootstrap'] = boot
        s['gate'] = stability_gate(cand, base, fd[s['best_candidate']], fd[s['strongest_baseline']], boot, G)
    s['recommended_method'] = s['best_candidate'] if s['gate']['passed'] else s['strongest_baseline']
    s['recommendation_kind'] = 'candidate_passed_gate' if s['gate']['passed'] else 'strongest_baseline_reference_gate_failed'
    SEL[unit] = s
    print(unit, json.dumps(json_safe(s), ensure_ascii=False)[:1500])
POOLED = pd.concat(POOLED, ignore_index=True)
BYFOLD = pd.concat(BYFOLD, ignore_index=True)
ABL = pd.DataFrame(ABL)
write_csv(REP / 'oof_metrics_pooled.csv', POOLED)
write_csv(REP / 'oof_metrics_by_fold.csv', BYFOLD)
write_csv(REP / 'ablation_deltas.csv', ABL)

# selection lock: written BEFORE any 2024/2025 diagnostic; records the actual input/config/OOF/metric hashes
FN11_SHA = sha256_bytes(functions_cells_source(NB11).encode('utf-8'))
LOCK_HASHES = {'experiment_lock_sha256': sha256_file(LOCK_PATH), 'protocol_sha256': sha256_file(PROTOCOL_PATH), 'model_config_sha256': sha256_file(CFG_PATH),
               'preparation_manifest_sha256': sha256_file(PM_PATH), 'notebook11_functions_sha256': FN11_SHA}
LOCK_FILES = ([{'path': str((MODEL_DATA / OOF_FILES[u]['path']).relative_to(OUT)) if inside(MODEL_DATA, OUT) else str(MODEL_DATA / OOF_FILES[u]['path']),
                'sha256': OOF_FILES[u]['sha256']} for u in CFG['units']]
              + [{'path': str((REP / n).relative_to(OUT)), 'sha256': sha256_file(REP / n)} for n in ('oof_metrics_pooled.csv', 'oof_metrics_by_fold.csv', 'ablation_deltas.csv', 'fold_boundaries.csv')])
SEL_LOCK_PATH = REP / 'selection_lock.json'
SEL_LOCK_SHA = write_selection_lock(SEL_LOCK_PATH, {
    'producer': CFG['producer'], 'created_utc': utc_now(), 'rule': CFG['selection']['rule'], 'gate_config': G,
    'note': 'written after pooled-OOF selection and BEFORE any 2024/2025 diagnostic; later-year scores cannot change these choices',
    'hashes': LOCK_HASHES, 'preparation_data_outputs': [{'path': o['path'], 'sha256': o['sha256']} for o in PM['data_outputs']],
    'files': LOCK_FILES, 'units': SEL})
RUN['selection_lock_sha256'] = SEL_LOCK_SHA
print('selection lock', SEL_LOCK_PATH, SEL_LOCK_SHA)
print(POOLED[['unit', 'method_id', 'status', 'mean_poisson_deviance', 'mae', 'n_deviance_floored']].sort_values(['unit', 'mean_poisson_deviance']).to_string(index=False))

## F. Final fit และ diagnostic ปีหลัง

fit เฉพาะ Train ที่จบก่อน cutoff; 2024 = development diagnostic ที่เคยเห็นแล้ว, 2025 = reused/previously-seen — **ไม่ใช่ untouched test**; reload bundle แล้วเทียบ prediction

In [ ]:
# F. Final fit on the allowed pre-issue Train cells only (cutoff 2023-12-31 18:00 +07), 2024 development / 2025 reused diagnostics, bundles + reload check
verify_selection_lock(SEL_LOCK_PATH, SEL_LOCK_SHA, LOCK_HASHES, OUT)  # lock unchanged BEFORE any later-year label is read
CFG_BY_ID = {c['id']: c for c in CFG['model_configs']}
DIAG, DIAG_FILES, BUNDLES, FINAL_LOG, RELOAD = {}, {}, [], [], []
_t0 = time.perf_counter()
for unit in CFG['units']:
    fr, F_, keys = FRAMES[unit], FINAL[unit], list(IFACE[unit]['keys'])
    tr = fr.loc[F_['train_mask']]
    dmask = np.any(list(F_['diag_masks'].values()), axis=0)
    dg = fr.loc[dmask, list(dict.fromkeys(keys + ['target_date', 'split', 'y']))].copy()
    dfr = fr.loc[dmask]
    ytr = tr['y'].to_numpy(float)
    bdir = MODEL_DATA / 'bundles' / unit
    # baselines: frozen at the final cutoff (no 2024/2025 label updates)
    for spec in CFG['baselines'][unit]:
        mid = baseline_method_id(spec['id'])
        st = fit_baseline(spec, tr, cutoff=FINAL_CUTOFF)
        dg[mid] = validate_predictions(predict_baseline(st, dfr), mid)
        p = write_json(bdir / f'{mid}.json', st)
        BUNDLES.append({'unit': unit, 'method_id': mid, 'kind': 'baseline_state_json', 'path': str(p.relative_to(MODEL_DATA)), 'sha256': sha256_file(p)})
        FINAL_LOG.append({'unit': unit, 'method_id': mid, 'status': 'ok', 'seconds': None, 'n_train': len(tr), 'n_convergence_warnings': 0, 'warnings': ''})
    # best member of each candidate family (selected on pooled OOF and locked above)
    for fam, mid in SEL[unit]['family_best'].items():
        if mid is None:
            FINAL_LOG.append({'unit': unit, 'method_id': f'(no complete {fam} member)', 'status': 'not_fitted', 'seconds': None})
            continue
        meta = method_meta(mid, CFG['model_configs'])
        c, cols = CFG_BY_ID[meta['config_id']], VARIANTS[unit][meta['variant']]
        cc, nc = [x for x in cols if x in CATS[unit]], [x for x in cols if x not in CATS[unit]]
        pipe = make_pipeline(c, cc, nc)
        try:
            _, lg = logged_call(lambda: pipe.fit(tr[cc + nc], ytr), THREADS)
            dg[mid] = validate_predictions(pipe.predict(dfr[cc + nc]), mid)
        except Exception as e:  # recorded; no fallback model
            FINAL_LOG.append({'unit': unit, 'method_id': mid, 'status': 'failed', 'error': f'{type(e).__name__}: {e}'[:500]})
            continue
        bmeta = {'unit': unit, 'method_id': mid, 'family': fam, 'config': c, 'variant': meta['variant'], 'categorical': cc, 'numeric': nc,
                 'feature_columns_in_order': cc + nc, 'fit_cutoff_local': FINAL_CUTOFF.isoformat(), 'n_train': int(len(tr)), 'versions': VERSIONS}
        bp = bdir / f'{mid}.joblib'
        BUNDLES.append({'unit': unit, 'method_id': mid, 'kind': 'sklearn_pipeline_joblib', 'path': str(bp.relative_to(MODEL_DATA)),
                        'sha256': save_model_bundle(bp, pipe, bmeta), 'feature_columns': cc + nc})
        FINAL_LOG.append({'unit': unit, 'method_id': mid, 'status': 'ok', 'seconds': lg['seconds'], 'n_train': len(tr),
                          'n_convergence_warnings': lg['n_convergence_warnings'], 'warnings': ' | '.join(lg['warnings'])})
        print(f'  final {unit} {mid}: {lg["seconds"]}s warnings={lg["n_warnings"]}')
    DIAG[unit] = dg
    p = MODEL_DATA / f'diagnostic_predictions_{unit}.csv.gz'
    write_csv_gz(p, dg)
    DIAG_FILES[unit] = {'path': p.name, 'sha256': sha256_file(p), 'rows': int(len(dg))}
    # reload every saved artifact and compare predictions on a fixed supported sample of 2024 rows
    rc = CFG['reload_check']
    pool = np.flatnonzero(dg['split'].eq(rc['split']).to_numpy())
    pick = np.sort(np.random.default_rng(rc['seed']).choice(pool, size=min(rc['n_rows'], len(pool)), replace=False))
    sample = dfr.iloc[pick]
    for b in [b for b in BUNDLES if b['unit'] == unit]:
        path = MODEL_DATA / b['path']
        if b['kind'] == 'baseline_state_json':
            verify_file(path, b['sha256'], 'baseline state')
            again = predict_baseline(read_json_strict(path), sample)
        else:
            again = load_model_bundle(path, b['sha256'])['pipeline'].predict(sample[b['feature_columns']])
        RELOAD.append(dict(unit=unit, method_id=b['method_id'], **reload_consistency(dg[b['method_id']].to_numpy()[pick], again, rc['max_abs_diff'])))
RUN['timings']['final_fit_and_diagnostics_seconds'] = round(time.perf_counter() - _t0, 1)
RELOAD = pd.DataFrame(RELOAD)
assert RELOAD['pass'].all(), RELOAD[~RELOAD['pass']]

# diagnostic metrics (2024 development / 2025 reused): reported, never used to change the locked selection
DMET = []
for unit in CFG['units']:
    dg = DIAG[unit]
    for sp, g in dg.groupby('split', sort=False):
        for mid in [c for c in dg.columns if c.startswith('baseline_') or '__' in c]:
            DMET.append(dict(unit=unit, split=sp, method_id=mid, **method_meta(mid, CFG['model_configs']),
                             **metric_summary(g['y'], g[mid], FLOOR, deviance=not is_zero_method(mid))))
DMET = pd.DataFrame(DMET)
write_csv(REP / 'diagnostic_metrics.csv', DMET)
write_csv(REP / 'final_fit_log.csv', pd.DataFrame(FINAL_LOG))
write_csv(REP / 'reload_check.csv', RELOAD)
verify_selection_lock(SEL_LOCK_PATH, SEL_LOCK_SHA, LOCK_HASHES, OUT)  # and unchanged AFTER the diagnostics
RUN['selection_lock_verified_after_diagnostics'] = True
print(DMET[['unit', 'split', 'method_id', 'mean_poisson_deviance', 'mae', 'n']].to_string(index=False))
print(RELOAD.to_string(index=False))

## G. รายงานไทย, model card และ manifest (next_stage_interface สำหรับ notebook 12)

In [ ]:
# G. Thai experiment report, model card and model manifest with the machine-readable interface for notebook 12
def _f(x, nd=4):
    return 'null' if x is None or (isinstance(x, float) and not math.isfinite(x)) else (f'{x:.{nd}f}' if isinstance(x, float) else str(x))

RUN['timings']['total_seconds'] = round(time.perf_counter() - T_RUN0, 1)
PM_ = POOLED.set_index(['unit', 'method_id'])
md = ['# รายงานการทดลองโมเดล V2 (Notebook 11)', '',
      'สถานะ: งานวิจัยเท่านั้น | production_ready = false | operational = BLOCKED | ไม่ใช่การพยากรณ์ความเสี่ยงรายบุคคล/เส้นทาง', '',
      'ป้ายกำกับ = จำนวนบันทึก MOT ที่ถูกเลือก (เฉพาะเครือข่าย MOT ไม่ใช่อุบัติเหตุทั้งหมด). เซลล์ training_eligible = false เป็นป้ายที่ไม่ทราบ และถูกตัดออกจากทุกวิธีเหมือนกัน '
      '(ตัวกรองนี้ได้จากผลลัพธ์ ใช้แบบ offline เท่านั้น).', '',
      '## 1. วิธีทดลอง', '',
      f"- หน่วย daily (จังหวัด-วัน, หลัก) และ 6h (จังหวัด x บล็อก 6 ชม.) ทดลองแยกกัน เลือกผู้ชนะแยกกัน ไม่เทียบ error ข้ามหน่วย",
      f"- 4 fold ขยายตามเวลา {', '.join(f['name'] for f in CFG['folds'])}; fit ได้เฉพาะเซลล์ Train ที่ target_end_local <= เวลา issue ของวันแรกที่ประเมิน (วันก่อนหน้า 18:00 +07)",
      f"- 3 ตระกูล: Poisson (alpha 0.001/0.01), Random Forest (poisson), HGB (poisson) x 5 ชุด feature ({', '.join(CFG['feature_variants'])}); ค่า config คงที่ตาม protocol ไม่มีการ tune",
      '- preprocessing (one-hot/ordinal, imputer, scaler) fit จาก Train ของแต่ละ fold เท่านั้น; คอลัมน์ที่หายทั้งหมดใน Train ถูกเก็บไว้; ค่า feature ที่หายไม่ทำให้แถวหาย',
      '- baseline fit จาก Train เท่านั้น; recent90 ถูกตรึง ณ เวลาเริ่ม fold (ไม่อัปเดตด้วยป้ายของช่วงประเมิน)',
      f"- metric หลัก: mean Poisson deviance แบบ pooled OOF (floor {FLOOR:g} ใช้เฉพาะใน deviance และนับจำนวนครั้ง); baseline ศูนย์ไม่มี deviance จำกัด (null)", '']
for unit in CFG['units']:
    s = SEL[unit]
    md += [f'## 2.{CFG["units"].index(unit) + 1} หน่วย {unit}', '', '| วิธี | ตระกูล | deviance (OOF) | MAE | RMSE | bias | floored |', '|---|---|---|---|---|---|---|']
    show = [m for m in s['family_best'].values() if m] + [m for m in POOLED.loc[POOLED['unit'].eq(unit) & POOLED['family'].eq('baseline'), 'method_id']]
    for m in show:
        r = PM_.loc[(unit, m)]
        tag = ' **(strongest baseline)**' if m == s['strongest_baseline'] else (' **(best candidate)**' if m == s['best_candidate'] else '')
        md.append(f"| {m}{tag} | {r['family']} | {_f(r.get('mean_poisson_deviance'))} | {_f(r.get('mae'))} | {_f(r.get('rmse'))} | {_f(r.get('bias'))} | {_f(r.get('n_deviance_floored'))} |")
    g = s['gate']
    md += ['', f"- ตัวแทนดีที่สุดของแต่ละตระกูล: {s['family_best']}", f"- วิธีที่ล้มเหลว/ไม่ครบทุก fold: {s['failed_methods'] or 'ไม่มี'}",
           f"- stability gate (เทียบ strongest baseline): {'ผ่าน' if g['passed'] else 'ไม่ผ่าน'}"]
    for k, v in g.get('checks', {}).items():
        md.append(f"  - {k}: value={_f(v.get('value'))} threshold={v.get('threshold')} -> {'pass' if v['pass'] else 'FAIL'}")
    if s.get('bootstrap'):
        b = s['bootstrap']
        md.append(f"- bootstrap (สำรวจ, ไม่ปรับ multiple comparison): ผลต่าง deviance {_f(b['point'], 5)} CI95 [{_f(b['lower'], 5)}, {_f(b['upper'], 5)}] จาก {b['n_blocks']} บล็อก 7 วัน")
    md += [f"- **ข้อเสนอ: {s['recommended_method']}** ({s['recommendation_kind']})", '']
    if not g['passed'] and s['best_candidate']:
        md.append(f"- ตัวเลือกที่ดีที่สุดจาก 3 ตระกูล ({s['best_candidate']}) ยังไม่ผ่านเกณฑ์ข้างต้น จึงใช้ baseline เป็น reference; ไม่ได้อ้างว่าโมเดลชนะ")
    a = ABL[ABL['unit'].eq(unit) & ABL['status'].eq('ok')]
    md += ['', '### ผลของการเพิ่ม feature (identical keys; ลบ = ดีขึ้น, บวก = แย่ลง)', '', '| config | เพิ่ม | เทียบกับ | Δ deviance | relative | fold ที่ดีขึ้น | ผล |', '|---|---|---|---|---|---|---|']
    md += [f"| {r.config_id} | {r.variant} | {r.reference} | {r.delta_deviance:+.5f} | {r.relative_change:+.3%} | {r.folds_improved}/4 | {r.effect} |" for r in a.itertuples()]
    md += ['', f"- harm (deviance เพิ่ม): {int(a['effect'].eq('harm').sum())} คู่; benefit: {int(a['effect'].eq('benefit').sum())} คู่", '']
md += ['## 3. Diagnostic ปีหลัง (fit เฉพาะ Train ถึง cutoff 2023-12-31 18:00 +07; ไม่ refit, ไม่เปลี่ยนตัวเลือก)', '',
       '2024 = development diagnostic ที่เคยเห็นแล้ว; 2025 = reused/previously-seen diagnostic — **ไม่มีชุดใดเป็น untouched test**', '',
       '| หน่วย | split | วิธี | deviance | MAE | n |', '|---|---|---|---|---|---|']
for unit in CFG['units']:
    keep = set(m for m in SEL[unit]['family_best'].values() if m) | {SEL[unit]['strongest_baseline']}
    for r in DMET[DMET['unit'].eq(unit) & DMET['method_id'].isin(keep)].itertuples():
        md.append(f"| {unit} | {r.split} | {r.method_id} | {_f(r.mean_poisson_deviance)} | {_f(r.mae)} | {r.n} |")
md += ['', f"- เซลล์ Train ที่ถูกตัดเพราะสิ้นสุดหลัง cutoff: { {u: FINAL[u]['n_excluded_after_cutoff'] for u in FINAL} }",
       f"- reload ตรวจซ้ำ {len(RELOAD)} artifact บนตัวอย่าง 2024 คงที่: ผ่านทั้งหมด = {bool(RELOAD['pass'].all())}", '',
       '## 4. ข้อจำกัด', '',
       '- เวลาเผยแพร่จริงของรายงานอุบัติเหตุยังไม่ได้ยืนยัน: ขอบเวลา issue ไม่ได้พิสูจน์ว่าใช้งานจริงได้',
       '- อากาศ = พยากรณ์ JMA lead ~48 ชม. ที่จุดตัวแทนจังหวัด (ความล่าช้าการเผยแพร่ไม่ยืนยัน, ความครอบคลุมอาจไม่ครบ); จราจร = ค่าเฉลี่ย AADT จากจุดสำรวจ ไม่ใช่ exposure offset',
       '- ไม่มี feature ประวัติอุบัติเหตุ (pending scenario แยก); cohort v2 ต่างจาก v1 จึงไม่อ้างตัวเลขเหนือกว่า v1',
       '- bootstrap เป็นการสำรวจหลังเลือกโมเดล; การวิเคราะห์รายกลุ่ม/กราฟละเอียดอยู่ใน notebook 12', '',
       f"เวลารันรวม {RUN['timings']['total_seconds']} วินาที; timings: {RUN['timings']}"]
(REP / 'experiment_report_th.md').write_text('\n'.join(md) + '\n', 'utf-8')

card = ['# Model card V2 (ร่างงานวิจัย, Notebook 11)', '', '- ใช้งาน: วิจัยเท่านั้น, operational BLOCKED, ไม่ใช่ความเสี่ยงรายบุคคล/เส้นทาง',
        '- ข้อมูล: panel v2 ที่ตรวจรับแล้วจาก notebook 10 (hash ล็อกใน selection_lock.json); ป้าย = บันทึก MOT ที่ถูกเลือก', '']
for unit in CFG['units']:
    s = SEL[unit]
    card += [f'## {unit}', f"- วิธีที่แนะนำ: {s['recommended_method']} ({s['recommendation_kind']})", f"- strongest baseline: {s['strongest_baseline']}",
             f"- ตัวแทนแต่ละตระกูล: {s['family_best']}", f"- gate ผ่าน: {s['gate']['passed']}", '']
card += ['## ข้อควรระวัง', '- 2024/2025 เคยเห็นแล้ว ไม่ใช่ test อิสระ; ห้ามเลือกใหม่จากผลปีหลัง', '- ตัวกรอง eligible ได้จากผลลัพธ์ ใช้ล่วงหน้าไม่ได้',
         '- traffic ไม่ใช่ตัวหาร exposure; weather มีความไม่แน่นอนเรื่องการเผยแพร่ย้อนหลัง']
(REP / 'model_card_th.md').write_text('\n'.join(card) + '\n', 'utf-8')

MAN = {
    'producer': CFG['producer'], 'schema': 'roadaware-v2-model-manifest/1', 'status': 'completed', 'status_note': 'research experiment; passing checks is not acceptance',
    'production_ready': False, 'operational_status': 'BLOCKED', 'started_utc': RUN['started_utc'], 'finished_utc': utc_now(),
    'functions_cells_sha256': FN11_SHA, 'code_cells_sha256': code_cells_sha256(NB11), 'inputs': INPUTS, 'versions': VERSIONS, 'threads': THREADS,
    'roots': {'prep_dir': str(PREP_DIR), 'prep_data': str(PREP_DATA), 'model_data': str(MODEL_DATA), 'output': str(OUT)},
    'timings': RUN['timings'], 'final_fit': RUN['final_fit'], 'source_run_check': SRC_RUN_CHECK,
    'selection_lock': {'path': SEL_LOCK_PATH.name, 'sha256': SEL_LOCK_SHA, 'verified_after_diagnostics': RUN['selection_lock_verified_after_diagnostics']},
    'selection': SEL, 'model_data_outputs': [dict(v, unit=u, kind='oof') for u, v in OOF_FILES.items()] + [dict(v, unit=u, kind='diagnostic') for u, v in DIAG_FILES.items()],
    'bundles': BUNDLES,
    'report_outputs': [{'path': p.name, 'sha256': sha256_file(p)} for p in sorted(REP.iterdir()) if p.is_file() and p.name != 'model_manifest.json'],
    'next_stage_interface': {
        'oof_predictions': {u: dict(OOF_FILES[u], root='model_data', keys=list(IFACE[u]['keys']), fold_column='fold', target='y',
                                    method_columns=[c for c in OOF[u].columns if c.startswith('baseline_') or '__' in c]) for u in CFG['units']},
        'diagnostic_predictions': {u: dict(DIAG_FILES[u], root='model_data', keys=list(IFACE[u]['keys']), split_column='split', target='y',
                                           method_columns=[c for c in DIAG[u].columns if c.startswith('baseline_') or '__' in c]) for u in CFG['units']},
        'panels_and_features': {u: {'panel': IFACE[u]['panel'], 'features': IFACE[u]['features'], 'root': 'prep_data', 'keys': list(IFACE[u]['keys']),
                                    'panel_sha256': PM_SHA[IFACE[u]['panel']], 'features_sha256': PM_SHA[IFACE[u]['features']],
                                    'join': 'join predictions to the accepted 10 features on these immutable keys to recover every fit-time feature'} for u in CFG['units']},
        'methods': {u: {m: dict(method_meta(m, CFG['model_configs']), feature_columns=(None if m.startswith('baseline_') else VARIANTS[u][method_meta(m, CFG['model_configs'])['variant']]))
                        for m in [c for c in OOF[u].columns if c.startswith('baseline_') or '__' in c]} for u in CFG['units']},
        'selected': {u: {k: SEL[u][k] for k in ('recommended_method', 'recommendation_kind', 'best_candidate', 'strongest_baseline', 'family_best')} for u in CFG['units']},
        'selection_lock': {'path': SEL_LOCK_PATH.name, 'sha256': SEL_LOCK_SHA}, 'deviance_floor': FLOOR,
        'diagnostic_split_labels': {s['name']: s['label_th'] for s in CFG['final_fit']['diagnostic_splits']},
        'deferred_to_12': 'plots, slice/calibration/residual analysis, block->daily reconciliation diagnostic'},
}
write_json(REP / 'model_manifest.json', MAN)
print('reports ->', REP, '| model data ->', MODEL_DATA, '| total seconds', RUN['timings']['total_seconds'])

In [ ]:
# Final checks: read-only 10 inputs and lock unchanged, selection lock verified, strict JSON, manifest hashes match the files on disk
for i in INPUTS:
    assert sha256_file(i['path']) == i['sha256'], f"input changed during the run: {i['path']}"
verify_experiment_lock(LOCK_PATH, PROTOCOL_PATH, CFG_PATH, PM_PATH, PM, IC['experiment_lock_schema'], IC['experiment_lock_fields'])
verify_selection_lock(SEL_LOCK_PATH, SEL_LOCK_SHA, LOCK_HASHES, OUT)
for p in sorted(REP.glob('*.json')):
    read_json_strict(p)
M = read_json_strict(REP / 'model_manifest.json')
assert all(sha256_file(REP / o['path']) == o['sha256'] for o in M['report_outputs'])
assert all(sha256_file(MODEL_DATA / o['path']) == o['sha256'] for o in M['model_data_outputs'])
assert all(sha256_file(MODEL_DATA / b['path']) == b['sha256'] for b in M['bundles'])
# independent recomputation of the pooled OOF deviance of each unit's recommended method from the saved predictions
for unit in CFG['units']:
    o = pd.read_csv(MODEL_DATA / OOF_FILES[unit]['path'], dtype={k: str for k in IFACE[unit]['keys']})
    m = SEL[unit]['recommended_method']
    d, _ = poisson_deviance_rows(o['y'], o[m], FLOOR)
    ref = PM_.loc[(unit, m), 'mean_poisson_deviance']
    assert abs(d.mean() - ref) <= 1e-12 * max(1.0, abs(ref)), (unit, m, d.mean(), ref)
assert M['production_ready'] is False and M['operational_status'] == 'BLOCKED'
print('final checks passed: inputs unchanged, experiment + selection locks verified, strict JSON, manifest hashes verified, OOF deviance recomputed;',
      f"{len(M['report_outputs'])} report files, {len(M['bundles'])} bundles")

## ข้อจำกัด

- เวลาเผยแพร่ของรายงานอุบัติเหตุยังไม่ยืนยัน; ขอบเวลาในโค้ดไม่ได้พิสูจน์การใช้งานจริง
- traffic ไม่ใช่ exposure offset; weather lead/latency ไม่ยืนยัน; ไม่มี feature ประวัติ (pending)
- ผลใด ๆ ในที่นี้ไม่อนุญาตการพยากรณ์เชิงปฏิบัติการ